In [6]:
from __future__ import annotations

import re
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [7]:
DATA_PATH = Path('data/resumes.csv')

sample_job_description = (
    'We are hiring a Data Analyst with experience in Python, SQL, Excel, pandas, visualization, '
    'statistics, data cleaning, machine learning basics, and dashboard reporting. The candidate should '
    'communicate clearly with stakeholders and transform raw data into business insights.'
)

sample_resumes = [
    {
        'candidate_name': 'Aisha Khan',
        'resume_text': 'Data analyst with 4 years of experience using Python, SQL, pandas, Excel, Power BI, '
                       'dashboard design, and data cleaning. Built reporting workflows and business metrics for sales teams.'
    },
    {
        'candidate_name': 'Rohan Mehta',
        'resume_text': 'Software engineer experienced in Java, Spring Boot, APIs, testing, Docker, and cloud deployment. '
                       'Worked on backend services and microservices.'
    },
    {
        'candidate_name': 'Sara Ali',
        'resume_text': 'Analytics intern with strong Excel, SQL, and data visualization skills. Familiar with Python notebooks, '
                       'statistics, data preparation, and presenting findings to non-technical teams.'
    },
    {
        'candidate_name': 'Daniel Kim',
        'resume_text': 'Marketing specialist with campaign analysis, SEO, content strategy, social media growth, and CRM reporting.'
    }
]

def load_resume_data(data_path: Path) -> pd.DataFrame:
    if data_path.exists():
        frame = pd.read_csv(data_path)
        required_columns = {'resume_text'}
        missing = required_columns - set(frame.columns)
        if missing:
            raise ValueError(f'Missing required columns in CSV: {sorted(missing)}')
        if 'candidate_name' not in frame.columns:
            frame['candidate_name'] = [f'Candidate {index + 1}' for index in range(len(frame))]
        return frame[['candidate_name', 'resume_text']].copy()

    return pd.DataFrame(sample_resumes)

resumes = load_resume_data(DATA_PATH)
job_description = sample_job_description

resumes

,candidate_name,resume_text
0,Aisha Khan,"Data analyst with 4 years of experience using Python, SQL, pandas, Excel, Power BI, dashboard design, and data clean..."
1,Rohan Mehta,"Software engineer experienced in Java, Spring Boot, APIs, testing, Docker, and cloud deployment. Worked on backend s..."
2,Sara Ali,"Analytics intern with strong Excel, SQL, and data visualization skills. Familiar with Python notebooks, statistics, ..."
3,Daniel Kim,"Marketing specialist with campaign analysis, SEO, content strategy, social media growth, and CRM reporting."


In [8]:
SKILL_LEXICON = {
    'python', 'sql', 'excel', 'pandas', 'numpy', 'statistics', 'data cleaning', 'data visualization',
    'dashboard', 'dashboards', 'power bi', 'tableau', 'machine learning', 'ml', 'reporting', 'business intelligence',
    'communication', 'stakeholder', 'stakeholders', 'analytics', 'analysis', 'data analysis', 'visualization',
    'modeling', 'models', 'feature engineering', 'data preparation'
}

STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'has', 'have', 'in', 'is', 'it', 'of', 'on',
    'or', 'that', 'the', 'this', 'to', 'with', 'we', 'you', 'your', 'their', 'they', 'our', 'was', 'were', 'will',
    'can', 'should', 'would', 'such', 'into', 'than', 'then', 'them', 'who', 'what', 'which', 'when', 'where', 'why', 'how'
}

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s+]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_skills(text: str, lexicon: set[str]) -> list[str]:
    cleaned = clean_text(text)
    matched = []
    for skill in sorted(lexicon, key=len, reverse=True):
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, cleaned):
            matched.append(skill)
    return matched

def tokenize_keywords(text: str) -> list[str]:
    tokens = re.findall(r'[a-z]+', clean_text(text))
    return [token for token in tokens if token not in STOPWORDS and len(token) > 2]

job_skills = extract_skills(job_description, SKILL_LEXICON)
job_keywords = tokenize_keywords(job_description)

job_skills, job_keywords[:20]

(['machine learning',
  'data cleaning',
  'visualization',
  'stakeholders',
  'statistics',
  'dashboard',
  'reporting',
  'python',
  'pandas',
  'excel',
  'sql'],
 ['hiring',
  'data',
  'analyst',
  'experience',
  'python',
  'sql',
  'excel',
  'pandas',
  'visualization',
  'statistics',
  'data',
  'cleaning',
  'machine',
  'learning',
  'basics',
  'dashboard',
  'reporting',
  'candidate',
  'communicate',
  'clearly'])

In [9]:
@dataclass
class ScoreWeights:
    semantic: float = 0.55
    skill_coverage: float = 0.35
    bonus_overlap: float = 0.10

WEIGHTS = ScoreWeights()

def compute_rankings(resume_frame: pd.DataFrame, job_text: str, lexicon: set[str], weights: ScoreWeights) -> pd.DataFrame:
    working = resume_frame.copy()
    working['clean_resume_text'] = working['resume_text'].map(clean_text)
    working['matched_skills'] = working['resume_text'].map(lambda text: extract_skills(text, lexicon))
    working['matched_skill_count'] = working['matched_skills'].map(len)
    working['missing_skills'] = working['matched_skills'].map(lambda found: sorted(set(job_skills) - set(found)))
    working['missing_skill_count'] = working['missing_skills'].map(len)

    corpus = [clean_text(job_text)] + working['clean_resume_text'].tolist()
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    tfidf_matrix = vectorizer.fit_transform(corpus)
    similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    working['semantic_similarity'] = similarity_scores

    required_skill_count = max(len(job_skills), 1)
    working['skill_coverage'] = working['matched_skill_count'] / required_skill_count

    job_overlap_keywords = set(tokenize_keywords(job_text))
    working['overlap_bonus'] = working['resume_text'].map(
        lambda text: len(job_overlap_keywords.intersection(tokenize_keywords(text))) / max(len(job_overlap_keywords), 1)
    )

    working['raw_score'] = (
        weights.semantic * working['semantic_similarity'] +
        weights.skill_coverage * working['skill_coverage'] +
        weights.bonus_overlap * working['overlap_bonus']
    )

    working['score'] = (working['raw_score'] * 100).round(2)
    working = working.sort_values(['score', 'semantic_similarity', 'matched_skill_count'], ascending=False).reset_index(drop=True)
    working['rank'] = np.arange(1, len(working) + 1)
    return working[['rank', 'candidate_name', 'score', 'semantic_similarity', 'skill_coverage', 'matched_skills', 'missing_skills', 'resume_text']]

ranked_candidates = compute_rankings(resumes, job_description, SKILL_LEXICON, WEIGHTS)
ranked_candidates

,rank,candidate_name,score,semantic_similarity,skill_coverage,matched_skills,missing_skills,resume_text
0,1,Aisha Khan,43.03,0.236227,0.727273,"[data cleaning, dashboard, reporting, power bi, python, pandas, excel, sql]","[machine learning, stakeholders, statistics, visualization]","Data analyst with 4 years of experience using Python, SQL, pandas, Excel, Power BI, dashboard design, and data clean..."
1,2,Sara Ali,36.72,0.159435,0.727273,"[data visualization, data preparation, visualization, statistics, analytics, python, excel, sql]","[dashboard, data cleaning, machine learning, pandas, reporting, stakeholders]","Analytics intern with strong Excel, SQL, and data visualization skills. Familiar with Python notebooks, statistics, ..."
2,3,Daniel Kim,8.91,0.038708,0.181818,"[reporting, analysis]","[dashboard, data cleaning, excel, machine learning, pandas, python, sql, stakeholders, statistics, visualization]","Marketing specialist with campaign analysis, SEO, content strategy, social media growth, and CRM reporting."
3,4,Rohan Mehta,1.81,0.032910,0.000000,[],"[dashboard, data cleaning, excel, machine learning, pandas, python, reporting, sql, stakeholders, statistics, visual...","Software engineer experienced in Java, Spring Boot, APIs, testing, Docker, and cloud deployment. Worked on backend s..."


In [10]:
def summarize_candidate(row: pd.Series) -> str:
    matched = ', '.join(row['matched_skills']) if row['matched_skills'] else 'none'
    missing = ', '.join(row['missing_skills']) if row['missing_skills'] else 'none'
    return (
        f"Rank {row['rank']}: {row['candidate_name']} | Score {row['score']} | "
        f"Matched: {matched} | Missing: {missing}"
    )

for _, row in ranked_candidates.iterrows():
    print(summarize_candidate(row))

print('\nJob skills detected:', ', '.join(job_skills))

Rank 1: Aisha Khan | Score 43.03 | Matched: data cleaning, dashboard, reporting, power bi, python, pandas, excel, sql | Missing: machine learning, stakeholders, statistics, visualization
Rank 2: Sara Ali | Score 36.72 | Matched: data visualization, data preparation, visualization, statistics, analytics, python, excel, sql | Missing: dashboard, data cleaning, machine learning, pandas, reporting, stakeholders
Rank 3: Daniel Kim | Score 8.91 | Matched: reporting, analysis | Missing: dashboard, data cleaning, excel, machine learning, pandas, python, sql, stakeholders, statistics, visualization
Rank 4: Rohan Mehta | Score 1.81 | Matched: none | Missing: dashboard, data cleaning, excel, machine learning, pandas, python, reporting, sql, stakeholders, statistics, visualization

Job skills detected: machine learning, data cleaning, visualization, stakeholders, statistics, dashboard, reporting, python, pandas, excel, sql
